## **Introduction to Integer Programming and Applications with Julia**

<table>
  <tr>
    <td>Chapter</td>
    <td>3 - Location Problems</td>
  </tr>
  <tr>
    <td>Author(s)</td>
    <td>Luiz Henrique Nogueira Lorena</td>
  </tr>
</table>

---

## Exercise 3.1

Formulate the P-Median Problem (PMP-N) and solve for 4 medians.

In [1]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # Data handling
using Distances # Distance computations

# Include utility functions for plotting the solution
include("utils/pmp-n_utils.jl")

# Function to solve the P-Median Problem with n clients and p facilities
function solve_pmp_n(file_path; p = 2)
    # Load latitude and longitude for all points
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)
    
    # Problem dimensions
    n = size(coordinates, 1)
    
    # Index sets
    I, J = 1:n, 1:n

    # Use Haversine distance to compute distance matrix
    D = Distances.pairwise(Distances.Haversine(), coordinates, dims=1)

    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Decision variables
    @variable(model, x[i in I, j in J], Bin)

    # Objective: minimize total distance
    @objective(model, Min, sum(D[i, j] * x[i, j] for i in I, j in J))

    # Constraint: exactly p facilities selected
    @constraint(model, sum(x[j, j] for j in J) == p)

    # Constraint: each client assigned to one facility
    @constraint(model, [i in I], sum(x[i, j] for j in J) == 1)

    # Constraint: assignment only to open facilities
    @constraint(model, [i in I, j in J], x[i, j] <= x[j, j])

    # Solve the model
    JuMP.optimize!(model)

    # Extract solution
    x_value = JuMP.value.(x)
    objective_value = JuMP.objective_value(model)

    # Print objective value
    println("Objective value: $objective_value meters")

    # Print selected facilities
    facility_ids = findall(x_value[j, j] .> 0.5 for j in J)
    println("Selected Facilities (p=$p): $facility_ids")

    # Print total client assignments for each selected facility
    assignments = Dict()
    for facility_id in facility_ids
        clients = findall(x_value[i, facility_id] .> 0.5 for i in I)
        assignments[facility_id] = clients
        println("Facility: $facility_id | $(length(clients)) clients assigned")
    end

    # Plot the solution
    fmap = plot_solution(coordinates, assignments, p)
    display(fmap)
end

# Example usage
solve_pmp_n("data/coordinates.csv", p = 4)

Objective value: 16528.64615458511 meters
Selected Facilities (p=4): [18, 78, 92, 99]
Facility: 18 | 24 clients assigned
Facility: 78 | 23 clients assigned
Facility: 92 | 23 clients assigned
Facility: 99 | 32 clients assigned


Python: <folium.folium.Map object at 0x7f29e73e2cf0>

---

## Exercise 3.2

Formulate the problem Fault-tolerant P-Median Problem [36] and solve for 3 medians (𝑝) and 2 for the redundancy parameter (𝑟).

$$
\begin{align}
\min \quad &
\sum_{i \in I} \sum_{j \in J} d_{ij} x_{ij}
\\[1ex]
\text{s.t.} \quad
& \sum_{j \in J} x_{jj} = p
\\[1ex]
& \sum_{j \in J} x_{ij} = r
&& \forall i \in I
\\[1ex]
& x_{ij} \le x_{jj}
&& \forall i \in I,\; j \in J
\\[1ex]
& x_{ij} \in \{0,1\}
&& \forall i \in I,\; j \in J
\\[1ex]
\end{align}
$$

[36] I. Vasilyev, A. V. Ushakov, N. Maltugueva, and A. Sforza, “An effective heuristic for large-scale fault-tolerant k-median problem,” Soft Computing, vol. 23, no. 9, pp. 2959–2967, 2019, doi: 10.1007/s00500-018-3130-0.

In [2]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # Data handling
using Distances # Distance computations

# Importing Python libraries for plotting
include("utils/pmp-fault_utils.jl")

# Function to solve the Fault-tolerant P-Median Problem
function solve_fault_tolerant_pmp(file_path; p = 3, r = 2)
    # Load latitude and longitude for all points
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)
    
    # Problem dimensions
    n = size(coordinates, 1)

    # Index sets
    I, J = 1:n, 1:n

    # Use Haversine distance to compute distance matrix
    D = Distances.pairwise(Distances.Haversine(), coordinates, dims=1)

    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Decision variables
    @variable(model, x[i in I, j in J], Bin)

    # Objective: minimize total distance
    @objective(model, Min, sum(D[i, j] * x[i, j] for i in I, j in J))

    # Constraint: exactly p facilities selected
    @constraint(model, sum(x[j, j] for j in J) == p)

    # Constraint: assignment only to open facilities
    @constraint(model, [i in I, j in J], x[i, j] <= x[j, j])

    # Constraint: each node i must be covered r times (including possibly by itself)
    @constraint(model, [i in I], sum(x[i, j] for j in J) == r)

    # Solve the model
    JuMP.optimize!(model)

    # Extract solution
    x_value = JuMP.value.(x)
    objective_value = JuMP.objective_value(model)

    # Print objective value
    println("Objective value: $objective_value meters")

    # Print selected facilities
    facility_ids = findall(x_value[j, j] .> 0.5 for j in J)
    println("Selected Facilities (p=$p): $facility_ids")

    # Print total client assignments for each selected facility
    assignments = Dict()
    for facility_id in facility_ids
        clients = findall(x_value[i, facility_id] .> 0.5 for i in I)
        assignments[facility_id] = clients
        println("Facility: $facility_id | $(length(clients)) clients assigned")
    end

    # Plot the solution
    fmap = plot_solution(coordinates, assignments, p)
    display(fmap)
end

# Example usage
solve_fault_tolerant_pmp("data/coordinates.csv", p = 3, r = 2)

Objective value: 62426.06000659928 meters
Selected Facilities (p=3): [7, 12, 55]
Facility: 7 | 52 clients assigned
Facility: 12 | 50 clients assigned
Facility: 55 | 102 clients assigned


Python: <folium.folium.Map object at 0x7f29e2e4c190>

---

## Exercise 3.3

Formulate a coverage problem that finds the minimum number of centers that must be opened so that all other points are covered (SCLP), considering a coverage radius of 400 meters.

In [3]:
using JuMP  # Modeling language
using HiGHS # Solver
using CSV   # Data handling

# Include utility functions for plotting the solution
include("utils/sclp_utils.jl")

function solve_sclp(file_path; radius = 200)
    # Load coordinates
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)

    # Create distance matrix
    D = create_distance_matrix(coordinates)

    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Number of demand/facility points
    n = size(D, 1)

    # Index sets
    I, J = 1:n, 1:n

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[j in J], Bin)

    # Objective: minimize number of facilities opened
    @objective(model, Min, sum(x[j] for j in J))

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in I], sum(C[i,j] * x[j] for j in J) >= 1)

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vector
    x_vals = JuMP.value.(x)

    # Get indices of selected facilities
    selected_facilities = [j for j in J if x_vals[j] > 0.5]

    # Display solution
    println("Number of facilities opened: ", Int(JuMP.objective_value(model)))
    println("Selected facility indices: ", selected_facilities)

    # Display solution on map
    fmap = plot_solution(selected_facilities, coordinates, radius)
    display(fmap)
end

# Example usage
solve_sclp("data/coordinates.csv", radius = 400)

Number of facilities opened: 4
Selected facility indices: [1, 12, 50, 73]


Python: <folium.folium.Map object at 0x7f29e2858f50>

---

## Exercise 3.4

Formulate the SCLP mathematical model from question 3 considering the intersections (overlapping) between the covers, using the approach presented in the article [37], and a covering radius of 400 meters.

$$
\begin{align}
\min \quad &
\sum_{j \in J} x_{j} + u
\\[1ex]
\text{s.t.} \quad
& \sum_{j \in J} c_{ij} x_j \ge 1 && \forall i \in I
\\[1ex]
& \sum_{i \in I} \sum_{j \in J} (x_{i} + x_{j} - 1) \ |\Psi_{ij}-z| \leq u
\\[1ex]
& x_{j} \in \{0,1\}
&& \forall j \in J
\\[1ex]
& u \in \mathbb{R}
\\[1ex]
\end{align}
$$

The continuous variable $u$ is used to represent overlap penalties, $c_{ij}$ is a boolean coverage matrix that represents the if facility (j) covers point (i). Finally, $\Psi_{ij}$ represents the Jaccard Matrix, wich measures overlap between the points covered by facilities (i) and (j), and is calculated as: 

$$
\Psi_{ij} = \frac{|S_i\cap S_j|}{|S_i\cup S_j|}
$$

where $S_i$ is the set of points covered by a facility at point (i). Finnaly, $z$ is the overlap-control parameter:

- (z=0): Large overlap increases the penalty, spreading coverage zones and reducing unnecessary overlaps
- (z=1): Now highly overlapping facilities are favored.


#### Why they use ($x_i+x_j-1$)?

The authors try to control the pairwise similarity structure among selected facilities. The paper explicitly discusses this. They intentionally allow negative contributions when neither facility is selected. This creates a reward/penalty mechanism:

* selected pairs contribute positively,
* unselected pairs contribute negatively.

The authors use this to bias the optimization toward or against overlap patterns depending on (z). So although unconventional from a pure MILP modeling perspective, it is deliberate in their formulation.


[37] E. Araújo, A. Chaves, and L. Lorena, “A Mathematical Model for the Coverage Location Problem With Overlap Control,” Computers & Industrial Engineering, p. 106548, 2020, doi: 10.1016/j.cie.2020.106548.

In [4]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # Data handling
using Distances # Distance computations

# Include utility functions for plotting and distance matrix creation
include("utils/sclp_utils.jl")

# Function to solve SCLP with overlap control
function solve_sclp_overlap(file_path; radius = 200, z = 1)
    # Load coordinates
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)
    
    # Create distance matrix
    D = create_distance_matrix(coordinates)
    
    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Calculate jaccard's matrix between points coverages
    Ψ = Distances.pairwise(Jaccard(), eachrow(C))

    # Number of demand/facility points
    n = size(C, 1)

    # Index sets
    I, J = 1:n, 1:n

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[j in J], Bin)
    @variable(model, u)

    # Objective: minimize number of facilities opened
    @objective(model, Min, sum(x[j] for j in J) + u)

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in I], sum(C[i,j] * x[j] for j in J) >= 1)

    # Constraint: Jaccard distance condition
    @constraint(model, sum((x[i] + x[j] - 1) * abs(Ψ[i,j] - z) for i in I, j in J) <= u)

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vector
    x_vals = JuMP.value.(x)

    # Get indices of selected facilities
    selected_facilities = [j for j in J if x_vals[j] > 0.5]

    # Display solution
    println("Number of facilities opened: ", length(selected_facilities))
    println("Selected facility indices: ", selected_facilities)

    # Display solution on map
    fmap = plot_solution(selected_facilities, coordinates, radius)
    display(fmap)
end

# Example usage z = 0 (more overlap)
solve_sclp_overlap("data/coordinates.csv", radius = 400, z = 0)

Number of facilities opened: 4
Selected facility indices: [1, 7, 12, 73]


Python: <folium.folium.Map object at 0x7f29e748f360>

In [5]:
# Example usage z = 1 (less overlap)
solve_sclp_overlap("data/coordinates.csv", radius = 400, z = 1)

Python: <folium.folium.Map object at 0x7f29e72896e0>

Number of facilities opened: 4
Selected facility indices: [5, 43, 50, 97]


---

## Exercise 3.5

Formulate a coverage problem that locates exactly three centers that maximize the demand covered in the other centers (MCLP), considering a coverage radius of 400 meters.

In [6]:
using JuMP  # Modeling language
using HiGHS # Solver
using CSV   # For reading CSV files

# Include utility functions for plotting the solution
include("utils/mclp_utils.jl")

# Function to solve the MCLP given coverage radius and number of facilities
function solve_mclp(file_path; radius = 400, p = 3)
    # Load coordinates
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)

    # Create distance matrix
    D = create_distance_matrix(coordinates)

    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Number of demand/facility points
    n = size(C, 1)

    # Index sets
    I, J = 1:n, 1:n

    # Create a vector of equal demand for each point
    d = ones(n)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[j in J], Bin)
    @variable(model, y[i in I], Bin)

    # Objective: maximize covered demand
    @objective(model, Max, sum(d[i]*y[i] for i in I))

    # Constraint: exactly p facilities are opened
    @constraint(model, sum(x[j] for j in J) == p)

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in I], sum(C[i,j] * x[j] for j in J) >= y[i])

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vectors
    x_vals = JuMP.value.(x)
    y_vals = JuMP.value.(y)

    # Get indices of selected facilities
    selected_facilities = [j for j in J if x_vals[j] > 0.5]
    covered_points = [i for i in I if y_vals[i] > 0.5]

    # Calculate coverage percentage
    total_covered = length(covered_points)
    coverage_percentage = round(100*total_covered/n, digits=2)

    # Display solution
    println("Facilities opened: $selected_facilities")
    println("Points covered: $total_covered/$n ($coverage_percentage %)")

    # Display solution on map
    fmap = plot_solution(selected_facilities,covered_points, coordinates, radius)
    display(fmap)
end

# Example usage
solve_mclp("data/coordinates.csv", radius = 400, p = 3)

Facilities opened: [7, 40, 73]
Points covered: 101/102 (99.02 %)


Python: <folium.folium.Map object at 0x7f29e73120f0>

---

## Exercise 3.6

Formulate the MCLP mathematical model from question 5 considering the intersections (overlapping) between the covers, using the approach presented in the article [37], and a covering radius of 400 meters.

$$
\begin{align}
\max \quad &
\sum_{i \in I} y_{i} - u
\\[1ex]
\text{s.t.} \quad
& \sum_{i \in I} x_i = p
\\[1ex]
& \sum_{j \in J} c_{ij} x_j \ge y_i && \forall i \in I
\\[1ex]
& \sum_{i \in I} \sum_{j \in J} (x_{i} + x_{j} - 1) \ |\Psi_{ij}-z| \leq u
\\[1ex]
& x_{j}, y_i \in \{0,1\}
&& \forall i \in I, j \in J
\\[1ex]
& u \in \mathbb{R}
\\[1ex]
\end{align}
$$

[37] E. Araújo, A. Chaves, and L. Lorena, “A Mathematical Model for the Coverage Location Problem With Overlap Control,” Computers & Industrial Engineering, p. 106548, 2020, doi: 10.1016/j.cie.2020.106548.

In [7]:
using JuMP      # Modeling language
using HiGHS     # Solver
using CSV       # For reading CSV files
using Distances # Distance computations

# Include utility functions for plotting the solution
include("utils/mclp_utils.jl")

# Function to solve the MCLP with a given coverage radius and number of facilities
function solve_mclp(file_path; radius = 400, p = 3, z = 1)
    # Load coordinates
    coordinates = CSV.read(file_path, CSV.Tables.matrix, header=false)

    # Create distance matrix
    D = create_distance_matrix(coordinates)

    # Create coverage matrix considering the defined radius
    C = D .< radius

    # Calculate jaccard's matrix between points coverages
    Ψ = Distances.pairwise(Jaccard(), eachrow(C))

    # Number of demand/facility points
    n = size(C, 1)

    # Index sets
    I, J = 1:n, 1:n

    # Create a vector of equal demand for each point
    d = ones(n)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Variables
    @variable(model, x[j in J], Bin)
    @variable(model, y[i in I], Bin)
    @variable(model, u)

    # Objective: maximize covered demand
    @objective(model, Max, sum(d[i]*y[i] for i in I) - u)

    # Constraint: exactly p facilities are opened
    @constraint(model, sum(x[j] for j in J) == p)

    # Constraint: each demand point is covered by at least one facility
    @constraint(model, [i in I], sum(C[i,j] * x[j] for j in J) >= y[i])

    # Constraint: Jaccard distance condition
    @constraint(model, sum((x[i] + x[j] - 1) * abs(Ψ[i,j] - z) for i in I, j in J) <= u)

    # Solve the model
    JuMP.optimize!(model)

    # Get solution vectors
    x_vals = JuMP.value.(x)
    y_vals = JuMP.value.(y)

    # Get indices of selected facilities
    selected_facilities = [j for j in J if x_vals[j] > 0.5]
    covered_points = [i for i in I if y_vals[i] > 0.5]

    # Calculate coverage percentage
    total_covered = length(covered_points)
    coverage_percentage = round(100*total_covered/n, digits=2)

    # Display solution
    println("Facilities opened: $selected_facilities")
    println("Points covered: $total_covered/$n ($coverage_percentage %)")

    # Display solution on map
    fmap = plot_solution(selected_facilities,covered_points, coordinates, radius)
    display(fmap)
end

# Example usage z=0 (more overlap)
solve_mclp("data/coordinates.csv", radius = 400, p = 3, z = 0)

Facilities opened: [7, 38, 101]
Points covered: 99/102 (97.06 %)


Python: <folium.folium.Map object at 0x7f29e3027130>

In [8]:
# Example usage z=1 (less overlap)
solve_mclp("data/coordinates.csv", radius = 400, p = 3, z = 1)

Python: <folium.folium.Map object at 0x7f29e30268b0>

Facilities opened: [5, 60, 72]
Points covered: 54/102 (52.94 %)


---

## Exercise 3.7

Formulate the mathematical model known as TEAM model [37].

The paper The TEAM/FLEET Models for Simultaneous Facility and Equipment Siting extends classical covering models by jointly deciding:

1. where to locate facilities, and
2. how to allocate specialized equipment/teams to those facilities.

A common mixed-integer programming representation of the TEAM/FLEET model is the following.

$$
\begin{align}
\max \quad &
\sum_{i \in I} p_i y_{i}
\\[1ex]
\text{s.t.} \quad
& \sum_{j \in J} x_{j}^{a} = a
\\[1ex]
& \sum_{j \in J} x_{j}^{b} = b
\\[1ex]
& \sum_{j \in J} c_{ij}^{a} x_j^{a} \ge y_i && \forall i \in I
\\[1ex]
& \sum_{j \in J} c_{ij}^{b} x_j^{b} \ge y_i && \forall i \in I
\\[1ex]
& x_j^{a} \leq x_j^{b} && \forall j \in J
\\[1ex]
& x_{j}^{a}, x_{j}^{b} \in \{0,1\}
&& \forall j \in J
\\[1ex]
& y_{i} \in \{0,1\} && \forall i \in I
\\[1ex]
\end{align}
$$

where:

- $p_i$: region $i$ population
- $y_i$: variable that controls if point $i$ is covered by both ambulance types
- $x_{j}^{a}$: variable that controls if advanced care ambulance is placed in facility $j$
- $a$: Number of advanced care ambulance
- $x_{j}^{b}$: variable that controls if basic care ambulance is placed in facility $j$
- $b$: Number of basic care ambulance
- $c_{ij}^{a}$: covering of advanced ambulance
- $c_{ij}^{b}$: covering of basic ambulance

The coverage matrices are defined as:

$$
c^a_{ij} =
\begin{cases}
1 & \text{if } d_{ij} <= t^a \\
0 & \text{otherwise}
\end{cases}
\qquad
c^b_{ij} =
\begin{cases}
1 & \text{if } d_{ij} <= t^b \\
0 & \text{otherwise}
\end{cases}
$$

where:

- $d_{ij}$: distance is travel time from demand point $𝑖$ to facility $𝑗$ (duration matrix)
- $t^{a}$: advanced care maximum service time (minutes)
- $t^{b}$: basic care maximum service time (minutes)

In this model, a location for advanced care is selected only if basic care is selected too.

[37] D. Schilling, D. J. Elzinga, J. Cohon, R. Church, and C. ReVelle, “The TEAM/FLEET models for simultaneous facility and equipment siting,” Transportation Science, vol. 13, no. 2, pp. 163–175, 1979, doi: 10.1287/trsc.13.2.163.

In [9]:
using JuMP   # Modeling language
using HiGHS  # Solver
using HDF5   # For reading .h5 files
using Printf # For formatted printing

# Include utility functions for plotting
include("utils/team_utils.jl")

# Function to solve the ambulance location problem with basic and advanced care
function solve_team_model(file_path; basic_time = 15, basic_total = 4, 
                          advanced_time = 25, advanced_total = 2)

  # Load problem data from HDF5 file
  # - duration_matrix: matrix representing route duration
  # - polulation: vector of population at each demand point
  duration_matrix = h5read(file_path, "duration_matrix")
  population = h5read(file_path, "districts_population")

  # Problem parameters
  demands = length(population)         # Number of demand points
  facilities = size(duration_matrix,1) # Number of facilities

  # Index sets 
  F = 1:facilities
  D = 1:demands

  # Create coverage matrix for basic care
  C_B = duration_matrix .<= basic_time

  # Create coverage matrix for advanced care
  C_A = duration_matrix .<= advanced_time

  # Create model
  model = JuMP.Model(HiGHS.Optimizer)

  # Silent mode (solver output is not printed)
  JuMP.set_silent(model)

  # Variables
  @variable(model, xB[f in 1:facilities], Bin)
  @variable(model, xA[f in 1:facilities], Bin)
  @variable(model, y[d in 1:demands], Bin)

  # Objective: maximize covered population
  @objective(model, Max, sum(population[d] * y[d] for d in D))

  # Constraint: total number of ambulances for basic and advanced care
  @constraint(model, sum(xB) == basic_total)
  @constraint(model, sum(xA) == advanced_total)

  # Demand coverage constraints for basic care
  @constraint(model, 
              [d in D], 
              sum(C_B[f, d] * xB[f] for f in F) >= y[d])
  
  # Demand coverage constraints for advanced care
  @constraint(model, 
              [d in D], 
              sum(C_A[f, d] * xA[f] for f in F) >= y[d])

  # Advanced only if basic exists
  @constraint(model, [f in F], xA[f] <= xB[f])

  # Solve model
  JuMP.optimize!(model)

  # Extract results
  xB_opt = JuMP.value.(xB)
  xA_opt = JuMP.value.(xA)
  y_opt = JuMP.value.(y)
  z_opt = Int(JuMP.objective_value(model))

  # Calculate total covered population
  total_population = sum(population)
  coverage = round((z_opt / total_population) * 100, digits=2) 

  println("=== Optimization Results ===\n")
  println("Population covered by both types of ambulances: ", 
          "$z_opt / $total_population ($coverage%)")
  
  println("\n=== Facilities selected for each ambulance type ===\n")
  selected_basic = findall(xB_opt .> 0.5)
  println("Basic Care: $selected_basic")

  selected_advanced = findall(xA_opt .> 0.5)
  println("Advanced Care: $selected_advanced")

  println("\n=== Ambulances coverage by district ===\n")
  println("="^60)
  println(" District  |  Basic Care  |  Advanced Care  |  Population")
  println("="^60)
  
  for d in D
    percent = round((population[d]/total_population)*100, digits=2)
    if y_opt[d] < 0.5
      @printf("%10d | %12s | %15s | %5s (%2.2f%%)\n", d, 
              "Not Covered", "Not Covered",
              population[d], percent)
    else
      basic = [j for j in selected_basic if C_B[j, d]]
      advanced = [j for j in selected_advanced if C_A[j, d]]
      @printf("%10d | %12s | %15s | %5s (%2.2f%%)\n", d, 
              "$basic", "$advanced", 
              population[d], percent)
    end
  end

  # Plot the solution on a map
  fmap = plot_solution(file_path, xB_opt, xA_opt, y_opt)
  display(fmap)
end

# Example usage
solve_team_model("data/sjc_ambulances.h5",
                 basic_time = 15, basic_total = 4, 
                 advanced_time = 25, advanced_total = 2)

=== Optimization Results ===

Population covered by both types of ambulances: 639288 / 680044 (94.01%)

=== Facilities selected for each ambulance type ===

Basic Care: [4, 5, 8, 9]
Advanced Care: [4, 8]

=== Ambulances coverage by district ===

 District  |  Basic Care  |  Advanced Care  |  Population
         1 |          [5] |          [4, 8] | 40909 (6.02%)
         2 |  Not Covered |     Not Covered |   342 (0.05%)
         3 |       [4, 5] |          [4, 8] | 16012 (2.35%)
         4 |          [5] |          [4, 8] |   167 (0.02%)
         5 |       [4, 5] |          [4, 8] | 15544 (2.29%)
         6 |  Not Covered |     Not Covered |   696 (0.10%)
         7 |       [4, 5] |          [4, 8] | 21916 (3.22%)
         8 |       [4, 5] |          [4, 8] | 39506 (5.81%)
         9 |  Not Covered |     Not Covered |   536 (0.08%)
        10 |       [8, 9] |          [4, 8] | 32782 (4.82%)
        11 |    [4, 5, 8] |          [4, 8] | 14260 (2.10%)
        12 |          [8] |         

Python: <folium.folium.Map object at 0x7f29dd9b89e0>